## SEF Y MEF

Para cada segundo (fila) se calcula la potencia total (suma de la potencia de todas las frecuencias): P_total = 10 + 20 + 40 + 20 + 10 + ...

Se calcula la potencia acumulada: 
Frec1 = Pot1, Frec2 = Pot1 + Pot2, Frec3 = Pot1 + Pot2 + Pot3, ...


### SEF: primera frecuencia donde la acumulada alcanza el 95% de la potencia total.

Frecuencia donde la potencia acumulada alcanza el 95%. Percentil 95 de la distribución acumulada de la potencia espectral.
 - SEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.95


### MEF: primera frecuencia donde la acumulada alcanza el 50% de la potencia total.

Mediana de la distribución de potencia espectral.
 - MEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.50


Esto se debe calcular sobre potencia lineal, no sobre dB. 

pot_media = (
    df_dsa_ch1_lin[cols_freq].to_numpy(dtype=float) +
    df_dsa_ch2_lin[cols_freq].to_numpy(dtype=float)
) / 2

# Métricas y correlación

## Z-score
Se normalizan las matrices con z-score global antes de compararlas. Se busca comparar patrones relativos y no solo valores absolutos en dB.

Ambas matrices tienen escalas distintas ya que la exportada en el archivo .f_a parece ser fruto de un procesamiento o suavizado realizado en el propio monitor BIS.

Comparación de las figuras tras la normalización de colores/escala común: tras el z-score, los colores representan intensidad relativa dentro de cada matriz, no dB reales.

$$
Z(t,f)=\frac{D(t,f)-\mu_D}{\sigma_D}
$$

Donde: 
 - D(t,f) es el valor de la DSA en el instante \(t\) 
 - la frecuencia \(f\)
 - $\mu_D$ es la media global de la matriz 
 - $\sigma_D$ su desviación típica.
 

## Correlación
Medida estadística que indica cómo dos variables se relacionan entre sí. Suele ser un número que varía entre -1 y +1. Existen 3 tipos:
 - **Correlación positiva (+1):**  Ambas variables se mueven en la misma dirección. Si una variable aumenta, la otra también aumenta. 
 - **Correlación negativa (-1):** Las variables se mueven en direcciones opuestas. Si una variable aumenta, la otra disminuye.
 - **Sin correlación (0):** No existe ningún patrón predecible entre ambas. El comportamiento de una no afecta ni pr.edice el de la otra.


### Correlación por frecuencias
En lugar de comparar toda la matriz se compara columna a columna.
 - Para cada frecuencia calcula Pearson entre las dos series temporales.
 - En qué bandas la reconstrucción se parece más o menos al BIS.

### Correlación por tiempo
En lugar de comparar toda la matriz se compara segundo a segundo.
 - Para cada segundo calcula Pearson entre los vectores de frecuencias completos. 
 - En qué momentos se parecen más o menos.
 
 
## Métricas
1. Para calcularlas primero hay que asegurar que ambas matrices tengan las mismas columnas de frecuencia en el mismo orden y en el mismo rango de tiempo.

2. Como vamos a comparar matrices que están en escalas distintas (se ve cuando buscamos el valor mínimo y máximo de potencia en cada DSA nos salen valores alejados) se normalizan mediante el z-score global. Esto sirve para comparar patrones de similitud espectral.

3. Después, en cada matriz solo se seleccionan las posiciones donde ambas matrices tienen datos válidos. Una zona en blanco por valores NaN no se compara.

Al aplicar este filtro, ambas matrices se convierten en vectores de valores válidos de cada matriz. Gracias a esto, se pueden hacer cálculos comparando posiciones.

### MAE
Mide error absoluto medio. Se suele referir a los errores en un modelo de predicción, pero en este caso comparan dos matrices DSA.

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |A_i - B_i|$$

Donde:
 - i: cada una de las posiciones de los vectores
 - n: nº total de posiciones
 - A y B: son cada una de las matrices (ahora vectores)
 
Interpretación:
 - Si se comparan matrices normalizadas con z-score, un MAE de 0.56 significa que las celdas difieren unas 0.56 desviaciones típicas.
 - **Más bajo:** mejor
 - **Más alto:** peor


### RMSE

$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (A_i - B_i)^2}$$

Interpretación:
 - **Más bajo:** mejor ajuste global
  - **Mucho mayor que MAE:** hay errores puntuales grandes
 - También mide error, pero penaliza más los errores grandes porque primero eleva al cuadrado.


### BIAS

$$Bias = \frac{1}{n} \sum_{i=1}^{n} (B_i - A_i)$$

Media de la resta para ver hacia donde se dirige el error. Mide si una matriz tiende a estar sistemáticamente por encima o por debajo de la otra.

Interpretación:
 - bias > 0: entonces la matriz B, normalmente el .f_a, tiene valores mayores que la reconstruida.
 - bias < 0: entonces la reconstruida tiene valores mayores que el .f_a.
 
Como trabajamos con el z-score global en ambas matrices, el bias suele salir muy cerca de 0, porque ambas matrices han sido centradas respecto a su media.


### PEARSON $r$
Cuantificar la similitud lineal entre los valores normalizados de ambas matrices DSA.

En este caso, determina si, cuando una zona de la DSA reconstruida sube, también lo hace la zona equivalente del .f_a de forma proporcional o no.

$$r = \frac{\sum_{i=1}^{n} (A_i - \bar{A})(B_i - \bar{B})}{\sqrt{\sum_{i=1}^{n} (A_i - \bar{A})^2} \sqrt{\sum_{i=1}^{n} (B_i - \bar{B})^2}}$$

Donde:
 - n: número total de valores válidos comparados
 - Ai y Bi: son los valores individuales de las matrices original y reconstruida en la posición i
 - $\bar{A}$ y $\bar{B}$: son las medias globales de los valores de las matrices A y B respectivamente.

Interpretación: 
 - +1: relación lineal perfecta positiva
 - 0: sin relación lineal clara
 - -1: relación lineal perfecta negativa
 
Si Pearson aumenta después del suavizado, significa que la reconstruida se está pareciendo más al .f_a en términos de variación lineal de intensidad. No significa que sean idénticas, pero sí que los patrones varían de forma parecida.


### SPEARMAN $r_s$ o $\rho$
Evalúa la fuerza y dirección de la asociación entre los arrays. Evalúa si la estructura de intensidades se mantenía entre matrices aunque la relación no fuera estrictamente lineal. Compara rangos de los valores. Ayuda a saber si se respeta la distribución de potencia.

En este caso, nos indica si, cuando una zona es relativamente alta en la mtriz reconstruida, también suele ser alta en el .f_a o no.

$$r_s = 1 - \frac{6 \sum_{i=1}^{n} d_i^2}{n(n^2 - 1)}$$

Donde:
 - n: es el número total de valores válidos comparados.
 - di: es la diferencia entre la posición que ocupa un dato en la matriz A y la posición que ocupa su pareja en la matriz B.
     - Se extraen los valores de las matrices.
     - A cada valor se le otorga (de menor a mayor) un número del 1 al N. El 1 se otorga al más bajo, el 2 al siguiente, etc. Sin variar las posiciones de los valores en los arrays.
     - Entre cada par de posiciones (posición 0 de matriz A y posición 0 de matriz B) se hace la resta.

Interpretación:
 - $r_s \approx 1$: Relación monótona positiva perfecta. Si un valor crece en la matriz A, también crece en la matriz B, aunque no sea al mismo ritmo.
 - $r_s \approx 0$: No hay relación en el orden de los datos.
 - $r_s \approx -1$: Relación monótona negativa perfecta (cuando una crece, la otra siempre decrece).

# Suavizado

La diferencia entre la DSA reconstruida directamente desde EEG crudo y la DSA exportada por el BIS no se debe solo al cálculo espectral, sino también a que el propio monitor aplica una atenuación/suavizado temporal sobre el espectro de potencia antes de representar la DSA.

El propio archivo .spa incluye un campo **Spsmooth**, que afecta al espectro de potencia, y por consiguiente, a la matriz de densidad espectral.
 - Es la tasa de atenuación aplicada matemáticamente a los datos del espectro de potencia, de los cuales nace directamente la imagen de la DSA y sus variables derivadas (como la Frecuencia del Borde Espectral o SEF). 
 - Las opciones programadas en el equipo para promediar este suavizado espectral son 0, 5, 10, 30 o 60 segundos.
 
 
### Tercera matriz - solo suavizado temporal
La tercera matriz implementa un suavizado temporal inspirado en el parámetro SpSmooth documentado para el espectro BIS.


## En qué consiste
En este caso, significa aplicar un promedio temporal para que la DSA no cambie tan bruscamente de un segundo al siguiente.

Se trata de una media móvil temporal causal.

In [ ]:
"""
dsa_eeg_suav = dsa_eeg_comparacion.rolling(
    window=30,
    min_periods=1,
    center=False
).mean()


 - dsa_eeg_comparacion: representa la DSA reconstruida desde EEG crudo
                         -filas = tiempo
                         -columnas = frecuencias
                         -valores = intensidad/potencia espectral
 
 - .rolling: ventana móvil
             - recorre la matriz por grupos consecutivos de filas
             - una ventana de 30 filas equivale a unos 30 segundos
             
 - window=30: tamaño de ventana
          -Para calcular cada valor suavizado se usan hasta 30 valores temporales consecutivos.
          -Para una frecuencia concreta (8Hz):
              - valor suavizado en t=30 = media de los valores de 8 Hz desde t=1 hasta t=30
 
 - min_periods=1: permite hacer la media con menos valores para que no salga NaN
                 - permite calcular la media aunque al principio todavía no haya 30 valores disponibles
 
 - center=False: la ventana no está centrada en el punto actual, sino colocada hacia atrás
                 - Suavizado causal
                 - usa el valor actual y valores anteriores, pero no usa valores futuros.
                 - para calcular el valor en t = 30 s usa t = 1 s hasta t = 30 s (y no t = 15 s hasta t = 45 s)
                 
 - .mean(): la media aritmética se aplica dentro de cada ventana.
           - Cada valor se reemplaza por el promedio de los valores incluidos en su ventana temporal.

"""

Para cada frecuencia, cada valor se sustituye por la media de los valores anteriores dentro de una ventana temporal.

Si se usa una window = 30, el valor suavizado en el segundo actual se calcula aproximadamente como:

$$DSA_{\text{suavizada}}(t, f) = \frac{1}{N} \sum_{k=0}^{N-1} DSA(t-k, f)$$

Donde:
 - DSA_suav(t, f): valor suavizado de la DSA en ese mismo instante t y frecuencia f
 - DSA(t-k, f): es el valor original de la matriz en esa frecuencia $f$, pero $k$ instantes de tiempo hacia atrás.
 - t: representa el instante temporal actual
 - f: representa una frecuencia concreta de la DSA
 - N: nº segundos tamaño de la ventana
 - k: es el índice que recorre los segundos anteriores dentro de la ventana de suavizado
 
Es decir, el valor suavizado actual es la media del valor actual + el valor de cada uno de los 29 segundos anteriores. 

Esto se hace para cada frecuencia por separado.

### Problema de solo el suavizado temporal
Los manuales no utilizan la palabra retardo pero indican que 
 - Una tasa de atenuación más corta: proporciona una mayor capacidad de respuesta a los cambios de estado
 - Una tasa de atenuación más larga: proporciona una tendencia uniforme con menor variabilidad y sensibilidad a las interferencias.

El BIS trabaja con un procesamiento temporal interno: 
 - ventanas de 2 segundos (esto lo pone en el manual)
 - evaluación de calidad (también lo pone)
 - suavizado espectral SpSmooth (parámetro del spa)
 - variables procesadas del .spa

Por consiguiente, la DSA exportada en el archivo .f_a no es una imagen fiel y directa del EEG, sino una representación procesada del mismo.

El suavizado temporal causal utiliza la media del valor acutual y valores previos. Esto puede provocar que los cambios rápidos no aparezcan en el mismo instante que en una reconstrucción directa, sino con una respuesta más lenta o un desfase aparente.

# Shift temporal

Aquí se aplica el shift temporal como una prueba metodológica para evaluar si determinados patrones espectrales de la DSA reconstruida suavizada aparecen ligeramente adelantados o retrasados respecto a la DSA exportada por el BIS.

### Cuarta matriz - suavizado + shift
La cuarta matriz añade un desplazamiento temporal exploratorio para evaluar el retardo aparente producido por la atenuación temporal.

Las métricas muestran que ciertas combinaciones de suavizado y shift aumentan Pearson/Spearman y reducen MAE/RMSE. 
 - Algunos eventos espectrales quedan mejor alineados respecto a la f_a.

## En qué consiste
El shift no cambia la potencia, ni las frecuencias, ni los colores de la DSA. Solo cambia qué segundo de una matriz se compara con qué segundo de la otra.


In [ ]:
""" 
 - shift_final = 10: Definición del desplazamiento temporal. 
                     Desfase de 10 segundos entre la DSA suavizada y la línea temporal de referencia.



dsa_eeg_suav_shift = dsa_eeg_suav.iloc[:-shift_final].reset_index(drop=True): Crea una matriz nueva con suavizado + shift a partir de la suavizada anterior.

 - dsa_eeg_suav: DSA reconstruida a la que se le ha aplicado el suavizado temporal
 
 - .iloc[:-shift_final]:selecciona todas las filas salvo las últimas (últimos 10 segundos)
                        * Al desplazar la DSA en el tiempo, la matriz nueva tendrá la misma longitud que el nuevo vector temporal.
 
 - reset_index(drop=True): reinicia el índice de filas después del recorte.
                          * drop=True: evita conservar el índice antiguo como una nueva columna.
                          * Así la matriz queda con índices ordenados desde 0 hasta el final.
 
 
tiempo_opt_desfase = tiempo_eeg.iloc[shift_final:].reset_index(drop=True): Crea un nuevo vector temporal

 - tiempo_eeg: vector temporal asociado a la DSA reconstruida desde EEG
              * Contiene los instantes reales de cada fila de la matriz.
 
 - .iloc[shift_final:]: selecciona el vector temporal eliminando los primeros shift_final segundos.
                       * El nuevo tiempo empieza 10 segundos más tarde que el original.

 - .reset_index(drop=True): reinicia el índice del vector temporal después del recorte.
                           * El eje temporal volverá a empezar en 0 y encajará con la matriz que también se ha recortado.
 
--------------------------------------------------------------------------------------------------------------------------- 
Las dos señales tienen un patrón parecido, pero una parece ir un poco adelantada o retrasada respecto a la otra.

Al mover el eje temporal unos segundos nos quedamos con:
 - una parte de matriz sin temporalidad
 - unos segundos sin matriz
 
Eso se recorta para que las longitudes sigan cuadrando y nos quedamos con lo del medio
--------------------------------------------------------------------------------------------------------------------------- 

df_merge_opt_desfase = df_merge_plot.iloc[shift_final:].reset_index(drop=True): creación de un nuevo dF merge (el que tiene las variables del spa) con el shift temporal.

 - df_merge_plot: dF con las variables del .spa alineadas con la línea temporal de la DSA
 
 - .iloc[shift_final:]: elimina las primeras filas del DataFrame para que sus variables queden alineadas con el nuevo vector temporal desplazado.
 
 - .res:et_index(drop=True): reinicia el índice después del recorte y evita guardar el índice antiguo como columna.



* Si la DSA desplazada se va a representar sobre una nueva escala temporal, la máscara también tiene que ir con ella

mask_comun_desfase = mask_comun.iloc[shift_final:].reset_index(drop=True): creación de una nueva máscara (basada en la común) desplazada el tiempo que se indique.
 
 - mask_comun: máscara (serie de Pandas) calculada sobre la temporalidad original. 
              * Marca qué segundos deben considerarse no válidos. 
              * Comparte la misma línea temporal que la DSA original del f_a y la DSA reconstruida directa desde EEG.
 
 - .iloc[shift_final:]: elimina las primeras shift_final filas de la máscara.
 
 - .reset_index(drop=True): reinicia el índice después del recorte y evita guardar el índice antiguo como columna.



dsa_eeg_suav_shift_plot = dsa_eeg_suav_shift.copy(): Copia de la DSA suavizada desplazada para preparar la visualización

dsa_eeg_suav_shift_plot.loc[mask_comun_desfase.values, :] = np.nan: aplica la máscara desplazada sobre la DSA suavizada con shift.

 - .loc[mask_comun_desfase.values, :]: selecciona las filas de la DSA (de inicio a fin) donde la máscara desplazada vale True.
 
 -= np.nan: sustituye esos valores por NaN. El colormap está configurado para pintar los NaN en blanco.
 

matriz_opt_desfase, vmin_opt_desfase, vmax_opt_desfase, norm_opt_desfase, cmap_opt_desfase = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_plot,
    gamma=0.25
)
 - matriz_opt_desfase: matriz para dibujar el plor
 
 - vmin_opt_desfase: mínimo de escala de color
 
 - vmax_opt_desfase: máximo de escala de color

 - norm_opt_desfase: normalización de color
 
 - cmap_opt_desfase: colormap
 
 - dsa_eeg_suav_shift_plot: matriz suavizada, desplazada y enmascarada
 
 - gamma=0.25: misma corrección visual de color que en la matriz suavizada sin shift


"""

Toma la DSA suavizada reconstruida desde EEG y la representa sobre una línea temporal desplazada 10 segundos. Para mantener la coherencia, se recortan las últimas filas de la matriz y las primeras filas del vector temporal, del DataFrame de variables y de la máscara. De esta forma, se evalúa si los patrones espectrales de la DSA reconstruida suavizada quedan mejor alineados con los patrones observados en la DSA original del archivo .f_a.



---------------------------------------------------------------------------------------------------------------------------

Para estudiar la posible existencia de un desfase aparente entre la DSA reconstruida suavizada y la DSA exportada por el BIS, se aplicó un desplazamiento temporal exploratorio. Para ello, se recortaron las últimas filas de la matriz suavizada y se desplazó la línea temporal eliminando los primeros segundos del vector de tiempo. La máscara de invalidez y las variables alineadas se recortaron de forma equivalente, garantizando que las bandas blancas y las curvas superpuestas permanecieran coherentes con la nueva referencia temporal. Este procedimiento no modifica los valores espectrales, sino únicamente su asignación temporal, y se emplea exclusivamente como herramienta de comparación metodológica.